### 2. Data Preprocessing

This notebook prepares the processed Sydney air quality data for exploratory analysis and subsequent modelling. The preprocessing decisions are based on the data-quality issues identified during the data inspection stage.

#### 2.1 Setup and Project Settings

In [1]:
# Import the libraries needed for data preprocessing.
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Set the main project folders.
processed_dir = Path("../data/processed")
results_dir = Path("../results")
tables_dir = results_dir / "tables"

# These are the seven stations retained during data inspection.
retained_stations = [
    "Camden",
    "Liverpool",
    "Parramatta North",
    "Penrith",
    "Prospect",
    "Randwick",
    "Rozelle"
]

print("Preprocessing setup complete.")

Preprocessing setup complete.


#### 2.2 Variables Used for Analysis

The analysis focuses on PM2.5 as the target variable and uses available meteorological and pollutant variables as explanatory variables.

In [3]:
# These are the variables identified for the project.
# PM2.5 is the target, while the other variables are potential predictors.

target_variable = "PM2.5"

predictor_variables = [
    "TEMP",
    "HUMID",
    "WSP",
    "WDR",
    "RAIN",
    "PM10",
    "NO2",
    "CO",
    "NO",
    "OZONE",
    "SO2"
]

print("Target variable:", target_variable)
print("Number of potential predictors:", len(predictor_variables))

Target variable: PM2.5
Number of potential predictors: 11


#### 2.3 Combine Variables Within Each Station

The processed files contain different variables for the same monitoring station. They are combined using the timestamp so that each row represents one hourly observation for a station.

##### 2.3.1 Check for Duplicate Timestamps

In [4]:
# Check whether any processed files contain duplicate timestamps.
# This is important before combining the variables because duplicate
# timestamps can create many-to-many merges.

duplicate_summary = []

for station in retained_stations:

    station_dir = processed_dir / station

    for file in station_dir.glob("*.xlsx"):

        df = pd.read_excel(file, header=2)

        # Create one datetime value from the separate Date and Time columns.
        df["Datetime"] = pd.to_datetime(
            df["Date"].astype(str) + " " + df["Time"].astype(str),
            errors="coerce",
            dayfirst=True
        )

        duplicate_count = df["Datetime"].duplicated().sum()

        duplicate_summary.append({
            "Station": station,
            "File": file.name,
            "Duplicate_Timestamps": duplicate_count
        })

duplicate_df = pd.DataFrame(duplicate_summary)

# Only display files where duplicate timestamps were found.
duplicate_flagged = duplicate_df[
    duplicate_df["Duplicate_Timestamps"] > 0
]

display(duplicate_flagged)

,Station,File,Duplicate_Timestamps
0,Camden,"Air Temperature, Relative Humidity.xlsx",2390
1,Camden,"NO2, CO, SO2.xlsx",2390
2,Camden,"Ozone, NO.xlsx",2390
3,Camden,"PM10, PM2.5, Ammonia.xlsx",2390
4,Camden,Rainfall.xlsx,2390
5,Camden,"Wind Speed, Wind Direction.xlsx",2390
6,Liverpool,"Air Temperature, Relative Humidity.xlsx",2390
7,Liverpool,"NO2, CO.xlsx",2390
8,Liverpool,"Ozone, NO.xlsx",2390
9,Liverpool,"PM2.5, Ammonia.xlsx",2390


##### 2.3.2 Inspect Duplicate Timestamp Records

In [5]:
# Inspect a few duplicate timestamps from one processed file.
# We want to check whether the duplicate rows contain the same
# measurements or whether they contain different values.

sample_file = (
    processed_dir
    / "Camden"
    / "Air Temperature, Relative Humidity.xlsx"
)

sample_df = pd.read_excel(sample_file, header=2)

# Create one datetime value from the separate Date and Time columns.
sample_df["Datetime"] = pd.to_datetime(
    sample_df["Date"].astype(str) + " " + sample_df["Time"].astype(str),
    errors="coerce",
    dayfirst=True
)

# Keep only rows where the timestamp occurs more than once.
duplicate_rows = sample_df[
    sample_df["Datetime"].duplicated(keep=False)
].sort_values("Datetime")

display(duplicate_rows.head(20))

,Date,Time,CAMDEN TEMP 1h average [°C],CAMDEN HUMID 1h average [%],Datetime
23,01/01/2020,24:00,19.7,67.8,NaT
47,02/01/2020,24:00,20.4,86.0,NaT
71,03/01/2020,24:00,22.5,76.8,NaT
95,04/01/2020,24:00,25.5,44.9,NaT
119,05/01/2020,24:00,18.5,69.6,NaT
143,06/01/2020,24:00,20.1,86.5,NaT
167,07/01/2020,24:00,NaN,NaN,NaT
191,08/01/2020,24:00,21.6,68.1,NaT
215,09/01/2020,24:00,21.8,80.8,NaT
239,10/01/2020,24:00,25.9,67.2,NaT


##### 2.3.3 Combine Variables Within Each Station

In [6]:
# Combine all available variables into one hourly dataset for each station.
# The timestamp handling follows the approach already used in Notebook 01,
# including the conversion of "24:00" to midnight on the following day.

station_data = {}

for station in retained_stations:

    station_dir = processed_dir / station
    file_data = []

    for file in station_dir.glob("*.xlsx"):

        # Read the processed Excel file.
        df = pd.read_excel(file, header=2)

        # Identify the date and time columns.
        date_col = "Date"
        time_col = "Time"

        # Convert the date column to datetime.
        df[date_col] = pd.to_datetime(
            df[date_col],
            format="%d/%m/%Y",
            errors="coerce"
        )

        # Convert the time column to text.
        df[time_col] = df[time_col].astype(str).str.strip()

        # Handle "24:00" in the same way as Notebook 01.
        midnight_24 = df[time_col] == "24:00"

        df.loc[midnight_24, time_col] = "00:00"

        df.loc[midnight_24, date_col] = (
            df.loc[midnight_24, date_col]
            + pd.Timedelta(days=1)
        )

        # Create one combined timestamp.
        df["Datetime"] = pd.to_datetime(
            df[date_col].dt.strftime("%d/%m/%Y")
            + " "
            + df[time_col],
            format="%d/%m/%Y %H:%M",
            errors="coerce"
        )

        # Keep only the timestamp and measurement columns.
        measurement_columns = [
            col for col in df.columns
            if col not in [date_col, time_col, "Datetime"]
        ]

        df = df[["Datetime"] + measurement_columns]

        # Convert measurement columns to numeric values.
        for column in measurement_columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

        # Remove any rows where a valid timestamp could not be created.
        df = df.dropna(subset=["Datetime"])

        # Store this file for combining later.
        file_data.append(df)

    # Combine the files using Datetime rather than repeated many-to-many merges.
    if file_data:
        station_df = file_data[0].copy()

        for next_df in file_data[1:]:
            station_df = station_df.set_index("Datetime").join(
                next_df.set_index("Datetime"),
                how="outer"
            ).reset_index()

        # Sort the final station data chronologically.
        station_df = station_df.sort_values("Datetime").reset_index(drop=True)

        station_data[station] = station_df

        print(
            f"{station}: {station_df.shape[0]:,} rows, "
            f"{station_df.shape[1] - 1} measurement columns"
        )

print("\nAll retained station files have been combined.")

Camden: 57,384 rows, 11 measurement columns
Liverpool: 57,384 rows, 12 measurement columns
Parramatta North: 57,384 rows, 12 measurement columns
Penrith: 57,384 rows, 12 measurement columns
Prospect: 57,384 rows, 11 measurement columns
Randwick: 57,384 rows, 11 measurement columns
Rozelle: 57,384 rows, 12 measurement columns

All retained station files have been combined.


##### 2.3.4 Check Combined Station Data

In [7]:
# Check the structure of the combined data for each retained station.
# This confirms that the timestamps and measurement columns were
# combined correctly before we start cleaning the data.

for station in retained_stations:

    station_df = station_data[station]

    print(f"\n{station}")
    print("Shape:", station_df.shape)
    print("First timestamp:", station_df["Datetime"].min())
    print("Last timestamp:", station_df["Datetime"].max())

    display(station_df.head(3))


Camden
Shape: (57384, 12)
First timestamp: 2020-01-01 01:00:00
Last timestamp: 2026-07-19 00:00:00


,Datetime,CAMDEN TEMP 1h average [°C],CAMDEN HUMID 1h average [%],CAMDEN NO2 1h average [pphm],CAMDEN CO 1h average [ppm],CAMDEN NO 1h average [pphm],CAMDEN OZONE 1h average [pphm],CAMDEN PM10 1h average [µg/m³],CAMDEN PM2.5 1h average [µg/m³],CAMDEN RAIN 1h average [mm/m²],CAMDEN WDR 1h average [°],CAMDEN WSP 1h average [m/s]
0,2020-01-01 01:00:00,19.4,49.9,0.3,0.2,0.0,1.9,55.9,49.8,NaN,164.2,1.3
1,2020-01-01 02:00:00,18.5,59.3,NaN,NaN,NaN,NaN,35.3,31.4,NaN,179.5,1.6
2,2020-01-01 03:00:00,18.5,59.3,0.2,0.2,0.0,1.7,42.2,23.5,NaN,204.0,1.6



Liverpool
Shape: (57384, 13)
First timestamp: 2020-01-01 01:00:00
Last timestamp: 2026-07-19 00:00:00


,Datetime,LIVERPOOL TEMP 1h average [°C],LIVERPOOL HUMID 1h average [%],LIVERPOOL NO2 1h average [pphm],LIVERPOOL CO 1h average [ppm],LIVERPOOL NO 1h average [pphm],LIVERPOOL OZONE 1h average [pphm],LIVERPOOL PM2.5 1h average [µg/m³],LIVERPOOL RAIN 1h average [mm/m²],LIVERPOOL SO2 1h average [pphm],LIVERPOOL PM10 1h average [µg/m³],LIVERPOOL WDR 1h average [°],LIVERPOOL WSP 1h average [m/s]
0,2020-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Parramatta North
Shape: (57384, 13)
First timestamp: 2020-01-01 01:00:00
Last timestamp: 2026-07-19 00:00:00


...

,Datetime,PROSPECT TEMP 1h average [°C],PROSPECT HUMID 1h average [%],PROSPECT NO2 1h average [pphm],PROSPECT CO 1h average [ppm],PROSPECT NO 1h average [pphm],PROSPECT OZONE 1h average [pphm],PROSPECT PM2.5 1h average [µg/m³],PROSPECT SO2 1h average [pphm],PROSPECT PM10 1h average [µg/m³],PROSPECT WDR 1h average [°],PROSPECT WSP 1h average [m/s]
0,2020-01-01 01:00:00,19.3,66.2,0.2,0.1,-0.1,1.7,24.0,0.1,58.0,187.4,2.2
1,2020-01-01 02:00:00,19.2,68.1,NaN,NaN,NaN,NaN,17.2,NaN,37.2,201.6,2.2
2,2020-01-01 03:00:00,19.1,69.1,0.1,0.0,-0.1,1.8,9.0,0.0,39.8,196.8,2.2



Randwick
Shape: (57384, 12)
First timestamp: 2020-01-01 01:00:00
Last timestamp: 2026-07-19 00:00:00


,Datetime,RANDWICK TEMP 1h average [°C],RANDWICK HUMID 1h average [%],RANDWICK SO2 1h average [pphm],RANDWICK NO2 1h average [pphm],RANDWICK NO 1h average [pphm],RANDWICK OZONE 1h average [pphm],RANDWICK PM10 1h average [µg/m³],RANDWICK PM2.5 1h average [µg/m³],RANDWICK RAIN 1h average [mm/m²],RANDWICK WDR 1h average [°],RANDWICK WSP 1h average [m/s]
0,2020-01-01 01:00:00,19.5,73.0,0.0,0.0,-0.1,2.0,40.9,18.5,NaN,191.5,3.8
1,2020-01-01 02:00:00,19.0,74.6,NaN,NaN,NaN,NaN,45.4,8.1,NaN,214.1,3.6
2,2020-01-01 03:00:00,18.3,71.4,0.0,0.4,0.0,1.4,39.7,15.0,NaN,221.6,4.2



Rozelle
Shape: (57384, 13)
First timestamp: 2020-01-01 01:00:00
Last timestamp: 2026-07-19 00:00:00


,Datetime,ROZELLE TEMP 1h average [°C],ROZELLE HUMID 1h average [%],ROZELLE NO2 1h average [pphm],ROZELLE CO 1h average [ppm],ROZELLE NO 1h average [pphm],ROZELLE OZONE 1h average [pphm],ROZELLE PM2.5 1h average [µg/m³],ROZELLE RAIN 1h average [mm/m²],ROZELLE SO2 1h average [pphm],ROZELLE PM10 1h average [µg/m³],ROZELLE WDR 1h average [°],ROZELLE WSP 1h average [m/s]
0,2020-01-01 01:00:00,19.0,71.5,0.1,0.1,0.0,1.8,10.2,0.0,0.0,36.6,190.2,2.5
1,2020-01-01 02:00:00,19.0,71.6,NaN,NaN,NaN,NaN,10.9,0.0,NaN,35.2,188.9,2.2
2,2020-01-01 03:00:00,18.8,70.2,0.0,0.1,0.0,1.8,12.4,0.0,0.0,35.9,193.0,1.9


#### 2.4 Clean Invalid and Suspicious Values

##### 2.4.1 Convert Invalid Values to Missing

In [8]:
# Replace physically invalid measurements with NaN.
# We keep the rows themselves because other variables may still
# contain useful measurements at the same timestamp.

cleaned_station_data = {}

for station in retained_stations:

    df = station_data[station].copy()

    # Identify the measurement columns for this station.
    measurement_columns = [
        column for column in df.columns
        if column != "Datetime"
    ]

    # Apply physical validity rules to the measurement columns.
    for column in measurement_columns:

        column_upper = column.upper()

        # Humidity must be between 0% and 100%.
        if "HUMID" in column_upper:
            invalid = (df[column] < 0) | (df[column] > 100)

        # PM2.5 and PM10 concentrations cannot be negative.
        elif "PM2.5" in column_upper or "PM10" in column_upper:
            invalid = df[column] < 0

        # Wind direction must be between 0° and 360°.
        elif "WDR" in column_upper:
            invalid = (df[column] < 0) | (df[column] > 360)

        # Wind speed and rainfall cannot be negative.
        elif "WSP" in column_upper or "RAIN" in column_upper:
            invalid = df[column] < 0

        # Other pollutant concentrations cannot be negative.
        elif any(
            pollutant in column_upper
            for pollutant in ["NO2", "NO", "CO", "OZONE", "SO2"]
        ):
            invalid = df[column] < 0

        else:
            continue

        # Replace invalid measurements with NaN.
        df.loc[invalid, column] = np.nan

    cleaned_station_data[station] = df

print("Invalid measurements have been converted to NaN.")

Invalid measurements have been converted to NaN.


##### 2.4.2 Summary of Cleaned Invalid Values

In [9]:
# Count how many invalid measurements were converted to NaN
# for each station and variable.

cleaning_summary = []

for station in retained_stations:

    original_df = station_data[station]
    cleaned_df = cleaned_station_data[station]

    for column in original_df.columns:

        if column == "Datetime":
            continue

        # Count values that were present before cleaning
        # but are now missing.
        changed_to_nan = (
            original_df[column].notna()
            & cleaned_df[column].isna()
        ).sum()

        if changed_to_nan > 0:
            cleaning_summary.append({
                "Station": station,
                "Variable": column,
                "Values_Converted_to_NaN": changed_to_nan
            })

cleaning_summary_df = pd.DataFrame(cleaning_summary)

display(
    cleaning_summary_df.sort_values(
        "Values_Converted_to_NaN",
        ascending=False
    )
)

,Station,Variable,Values_Converted_to_NaN
9,Liverpool,LIVERPOOL CO 1h average [ppm],28387
25,Penrith,PENRITH CO 1h average [ppm],16458
33,Prospect,PROSPECT CO 1h average [ppm],13187
18,Parramatta North,PARRAMATTA NORTH NO 1h average [pphm],8301
2,Camden,CAMDEN CO 1h average [ppm],7299
6,Camden,CAMDEN PM2.5 1h average [µg/m³],6901
34,Prospect,PROSPECT NO 1h average [pphm],5167
35,Prospect,PROSPECT PM2.5 1h average [µg/m³],5056
28,Penrith,PENRITH PM2.5 1h average [µg/m³],4660
44,Randwick,RANDWICK PM2.5 1h average [µg/m³],4653


#### 2.5 Missing-Value Assessment After Cleaning


##### 2.5.1 Missing Values After Invalid-Value Cleaning

In [ ]:
# Check how much missing data remains after invalid measurements
# have been converted to NaN.

missing_after_cleaning = []

for station in retained_stations:

    df = cleaned_station_data[station]

    for column in df.columns:

        if column == "Datetime":
            continue

        missing_count = df[column].isna().sum()
        total_count = len(df)

        missing_percentage = (
            missing_count / total_count
        ) * 100

        missing_after_cleaning.append({
            "Station": station,
            "Variable": column,
            "Missing_Values": missing_count,
            "Total_Values": total_count,
            "Missing_Percentage": missing_percentage
        })

missing_after_cleaning_df = pd.DataFrame(
    missing_after_cleaning
)

#Displays full table
pd.set_option("display.max_rows", None)
display(
    missing_after_cleaning_df.sort_values(
        "Missing_Percentage",
        ascending=False
    )
)

,Station,Variable,Missing_Values,Total_Values,Missing_Percentage
14,Liverpool,LIVERPOOL CO 1h average [ppm],34628,57384,60.344347
38,Penrith,PENRITH CO 1h average [ppm],25478,57384,44.399136
66,Randwick,RANDWICK RAIN 1h average [mm/m²],24524,57384,42.736651
30,Parramatta North,PARRAMATTA NORTH RAIN 1h average [mm/m²],20625,57384,35.942074
50,Prospect,PROSPECT CO 1h average [ppm],17284,57384,30.119894
8,Camden,CAMDEN RAIN 1h average [mm/m²],15164,57384,26.425484
18,Liverpool,LIVERPOOL RAIN 1h average [mm/m²],15035,57384,26.200683
3,Camden,CAMDEN CO 1h average [ppm],13361,57384,23.283494
27,Parramatta North,PARRAMATTA NORTH NO 1h average [pphm],12553,57384,21.875436
41,Penrith,PENRITH PM2.5 1h average [µg/m³],10433,57384,18.181026


##### 2.5.2 Identify Identify Consecutive Missing-Value Gaps after converting to Invalid Values

In [13]:
# Find consecutive periods of missing values for each station and variable.
# This helps us distinguish short gaps, which may be suitable for
# interpolation, from long gaps, which should be left missing.

gap_summary = []

for station in retained_stations:

    df = cleaned_station_data[station].copy()

    for column in df.columns:

        if column == "Datetime":
            continue

        # Identify missing and non-missing observations.
        is_missing = df[column].isna()

        # Give each consecutive missing period a unique group number.
        gap_group = (
            is_missing.ne(is_missing.shift())
            .cumsum()
        )

        # Calculate the length of each missing period.
        gap_lengths = (
            df.loc[is_missing]
            .groupby(gap_group[is_missing])
            .size()
        )

        if len(gap_lengths) > 0:

            gap_summary.append({
                "Station": station,
                "Variable": column,
                "Number_of_Gaps": len(gap_lengths),
                "Longest_Gap_Hours": gap_lengths.max(),
                "Average_Gap_Hours": gap_lengths.mean()
            })

gap_summary_df = pd.DataFrame(gap_summary)

display(
    gap_summary_df.sort_values(
        "Longest_Gap_Hours",
        ascending=False
    )
)

,Station,Variable,Number_of_Gaps,Longest_Gap_Hours,Average_Gap_Hours
66,Randwick,RANDWICK RAIN 1h average [mm/m²],36,24037,681.222222
30,Parramatta North,PARRAMATTA NORTH RAIN 1h average [mm/m²],34,20487,606.617647
18,Liverpool,LIVERPOOL RAIN 1h average [mm/m²],73,14800,205.958904
8,Camden,CAMDEN RAIN 1h average [mm/m²],73,14488,207.726027
38,Penrith,PENRITH CO 1h average [ppm],2233,4442,11.409763
40,Penrith,PENRITH OZONE 1h average [pphm],2060,4442,3.385922
43,Penrith,PENRITH SO2 1h average [pphm],2085,4442,3.688249
45,Penrith,PENRITH WDR 1h average [°],46,4427,98.652174
46,Penrith,PENRITH WSP 1h average [m/s],46,4427,98.652174
41,Penrith,PENRITH PM2.5 1h average [µg/m³],2809,4384,3.714133


##### 2.5.3 Check if Missing Gaps Suitable for Interpolation

In [14]:
# Classify missing gaps as short or prolonged.
# Gaps of up to 7 days are considered short enough to consider
# for interpolation. Longer gaps will be left as missing.

gap_threshold = 168  # 7 days × 24 hours

interpolation_summary = []

for station in retained_stations:

    df = cleaned_station_data[station].copy()

    for column in df.columns:

        if column == "Datetime":
            continue

        is_missing = df[column].isna()

        gap_group = (
            is_missing.ne(is_missing.shift())
            .cumsum()
        )

        gap_lengths = (
            df.loc[is_missing]
            .groupby(gap_group[is_missing])
            .size()
        )

        if len(gap_lengths) == 0:
            continue

        short_gaps = (gap_lengths <= gap_threshold).sum()
        long_gaps = (gap_lengths > gap_threshold).sum()

        interpolation_summary.append({
            "Station": station,
            "Variable": column,
            "Total_Gaps": len(gap_lengths),
            "Short_Gaps_<=168h": short_gaps,
            "Long_Gaps_>168h": long_gaps,
            "Longest_Gap_Hours": gap_lengths.max()
        })

interpolation_summary_df = pd.DataFrame(
    interpolation_summary
)

display(
    interpolation_summary_df.sort_values(
        "Longest_Gap_Hours",
        ascending=False
    )
)

,Station,Variable,Total_Gaps,Short_Gaps_<=168h,Long_Gaps_>168h,Longest_Gap_Hours
66,Randwick,RANDWICK RAIN 1h average [mm/m²],36,35,1,24037
30,Parramatta North,PARRAMATTA NORTH RAIN 1h average [mm/m²],34,33,1,20487
18,Liverpool,LIVERPOOL RAIN 1h average [mm/m²],73,72,1,14800
8,Camden,CAMDEN RAIN 1h average [mm/m²],73,72,1,14488
38,Penrith,PENRITH CO 1h average [ppm],2233,2222,11,4442
40,Penrith,PENRITH OZONE 1h average [pphm],2060,2059,1,4442
43,Penrith,PENRITH SO2 1h average [pphm],2085,2083,2,4442
45,Penrith,PENRITH WDR 1h average [°],46,45,1,4427
46,Penrith,PENRITH WSP 1h average [m/s],46,45,1,4427
41,Penrith,PENRITH PM2.5 1h average [µg/m³],2809,2807,2,4384


##### 2.5.4 PM2.5 Missing-Gap Assessment

The missing-value patterns for PM2.5 are examined separately because PM2.5 is the primary target variable of this study. The distribution of missing-gap lengths is used to determine an appropriate interpolation limit while avoiding excessive estimation across prolonged periods of missing observations.

In [15]:
# Examine the distribution of PM2.5 missing-gap lengths for each retained station.

pm25_gap_details = []

for station in retained_stations:

    df = cleaned_station_data[station].copy()

    pm25_columns = [
        column for column in df.columns
        if "PM2.5" in column.upper()
    ]

    for column in pm25_columns:

        is_missing = df[column].isna()

        gap_group = is_missing.ne(is_missing.shift()).cumsum()

        gap_lengths = (
            df.loc[is_missing]
            .groupby(gap_group[is_missing])
            .size()
        )

        for gap_length in gap_lengths:
            pm25_gap_details.append({
                "Station": station,
                "Variable": column,
                "Gap_Length_Hours": gap_length
            })

pm25_gap_details_df = pd.DataFrame(pm25_gap_details)

display(
    pm25_gap_details_df.sort_values(
        ["Station", "Gap_Length_Hours"],
        ascending=[True, False]
    ).head(50)
)

,Station,Variable,Gap_Length_Hours
2625,Camden,CAMDEN PM2.5 1h average [µg/m³],530
2564,Camden,CAMDEN PM2.5 1h average [µg/m³],177
939,Camden,CAMDEN PM2.5 1h average [µg/m³],149
2036,Camden,CAMDEN PM2.5 1h average [µg/m³],96
3327,Camden,CAMDEN PM2.5 1h average [µg/m³],84
4189,Camden,CAMDEN PM2.5 1h average [µg/m³],77
3342,Camden,CAMDEN PM2.5 1h average [µg/m³],73
3336,Camden,CAMDEN PM2.5 1h average [µg/m³],66
1516,Camden,CAMDEN PM2.5 1h average [µg/m³],32
1085,Camden,CAMDEN PM2.5 1h average [µg/m³],30


##### 2.5.5 Selection of Interpolation Limit

Based on the observed missing-gap patterns, linear interpolation will be restricted to missing gaps of up to 48 consecutive hours. 

Longer gaps will remain missing to avoid introducing excessive estimated data across prolonged periods of unavailable observations.

In [16]:
# Summarise how many missing gaps would be interpolated using a 48-hour limit.

interpolation_limit = 48

interpolation_impact = []

for station in retained_stations:

    df = cleaned_station_data[station].copy()

    for column in df.columns:

        if column == "Datetime":
            continue

        is_missing = df[column].isna()

        gap_group = is_missing.ne(is_missing.shift()).cumsum()

        gap_lengths = (
            df.loc[is_missing]
            .groupby(gap_group[is_missing])
            .size()
        )

        if len(gap_lengths) == 0:
            continue

        gaps_to_interpolate = (gap_lengths <= interpolation_limit).sum()
        missing_values_to_interpolate = gap_lengths[
            gap_lengths <= interpolation_limit
        ].sum()

        interpolation_impact.append({
            "Station": station,
            "Variable": column,
            "Total_Missing_Gaps": len(gap_lengths),
            "Gaps_<=48h": gaps_to_interpolate,
            "Values_Eligible_for_Interpolation": missing_values_to_interpolate,
            "Longest_Gap_Hours": gap_lengths.max()
        })

interpolation_impact_df = pd.DataFrame(interpolation_impact)

display(
    interpolation_impact_df.sort_values(
        "Values_Eligible_for_Interpolation",
        ascending=False
    )
)

,Station,Variable,Total_Missing_Gaps,Gaps_<=48h,Values_Eligible_for_Interpolation,Longest_Gap_Hours
14,Liverpool,LIVERPOOL CO 1h average [ppm],2936,2794,19661,551
50,Prospect,PROSPECT CO 1h average [ppm],3535,3499,14304,207
38,Penrith,PENRITH CO 1h average [ppm],2233,2160,12000,4442
27,Parramatta North,PARRAMATTA NORTH NO 1h average [pphm],3451,3440,11542,183
3,Camden,CAMDEN CO 1h average [ppm],2702,2680,9198,2329
7,Camden,CAMDEN PM2.5 1h average [µg/m³],4231,4223,8689,530
51,Prospect,PROSPECT NO 1h average [pphm],3509,3500,8217,173
4,Camden,CAMDEN NO 1h average [pphm],3500,3492,7515,2056
73,Rozelle,ROZELLE NO 1h average [pphm],3147,3144,6529,373
53,Prospect,PROSPECT PM2.5 1h average [µg/m³],3146,3138,6133,850


##### 2.5.6 Short-Gap Linear Interpolation

Linear interpolation is applied to short missing-value gaps of up to 48 consecutive hours. 

Longer gaps are retained as missing because estimating values across prolonged periods may introduce substantial artificial variation. 

The interpolation is performed only between available observations, so gaps at the beginning or end of a series are not extrapolated.

**Rain and Wind Direction are not linearly interpolated here because their values do not behave as simple continuous linear variables.**
- Rainfall can occur as short-duration events, 
- wind direction is circular, with 0° and 360° representing the same direction. 

These variables are therefore retained for separate treatment where required.

In [22]:
# Interpolate only complete missing gaps of 48 hours or less.
# Longer gaps are left unchanged.
# Rainfall and wind direction are not interpolated.

import re

interpolated_station_data = {}

interpolation_results = []

interpolation_variables = [
    "PM2.5",
    "PM10",
    "CO",
    "NO",
    "NO2",
    "OZONE",
    "SO2",
    "TEMP",
    "HUMID",
    "WSP"
]

interpolation_limit = 48

for station in retained_stations:

    df = cleaned_station_data[station].copy()

    for column in df.columns:

        if column == "Datetime":
            continue

        column_upper = column.upper()

        # Identify the actual variable name in the column.
        # Word boundaries prevent "NO" from matching "NORTH".
        should_interpolate = any(
            re.search(
                rf"(?<![A-Z0-9]){re.escape(variable.upper())}(?![A-Z0-9])",
                column_upper
            )
            for variable in interpolation_variables
        )

        if not should_interpolate:
            continue

        original_missing = df[column].isna()

        # Identify each consecutive missing-value gap.
        gap_group = original_missing.ne(
            original_missing.shift()
        ).cumsum()

        gap_lengths = (
            df.loc[original_missing]
            .groupby(gap_group[original_missing])
            .size()
        )

        # Identify only gaps that are short enough to interpolate.
        short_gap_groups = gap_lengths[
            gap_lengths <= interpolation_limit
        ].index

        # Create a mask containing only missing values
        # belonging to short gaps.
        short_gap_mask = (
            original_missing
            & gap_group.isin(short_gap_groups)
        )

        # Temporarily interpolate the complete series.
        interpolated_series = df[column].interpolate(
            method="linear",
            limit_area="inside"
        )

        # Apply interpolated values only to eligible gaps.
        df.loc[short_gap_mask, column] = (
            interpolated_series.loc[short_gap_mask]
        )

        values_filled = (
            original_missing
            & df[column].notna()
        ).sum()

        interpolation_results.append({
            "Station": station,
            "Variable": column,
            "Values_Interpolated": values_filled
        })

    interpolated_station_data[station] = df

interpolation_results_df = pd.DataFrame(interpolation_results)

display(
    interpolation_results_df[
        interpolation_results_df["Values_Interpolated"] > 0
    ].sort_values(
        "Values_Interpolated",
        ascending=False
    )
)

print("Short-gap interpolation completed.")

,Station,Variable,Values_Interpolated
12,Liverpool,LIVERPOOL CO 1h average [ppm],19623
42,Prospect,PROSPECT CO 1h average [ppm],14304
32,Penrith,PENRITH CO 1h average [ppm],12000
23,Parramatta North,PARRAMATTA NORTH NO 1h average [pphm],11542
3,Camden,CAMDEN CO 1h average [ppm],9198
7,Camden,CAMDEN PM2.5 1h average [µg/m³],8689
43,Prospect,PROSPECT NO 1h average [pphm],8214
4,Camden,CAMDEN NO 1h average [pphm],7514
62,Rozelle,ROZELLE NO 1h average [pphm],6529
45,Prospect,PROSPECT PM2.5 1h average [µg/m³],6133


Short-gap interpolation completed.


###### Also, the fact that some variables have fewer values interpolated than the number of values that were eligible in 2.5.5 is fine. For example:

###### - Liverpool CO: 19,661 eligible → 19,623 actually interpolated
###### - Camden PM2.5: 8,689 eligible → 8,689 actually interpolated
###### - Penrith PM2.5: 5,740 eligible → 5,720 actually interpolated

###### That's because limit_area="inside" prevents interpolation when a gap is at the beginning or end of the available series. We don't extrapolate beyond observed data.


##### 2.5.7 Post-Interpolation Verification

The interpolated datasets are checked to confirm that short missing-value gaps have been filled as intended, while excluded variables and prolonged missing-value gaps remain unchanged. The checks also confirm that the interpolation process has not altered the hourly timestamp structure.

In [24]:
# Verify the results of short-gap interpolation.

#Section 1: Timestamp and row-count check
print("=== 1. Timestamp and row-count check ===")

for station in retained_stations:

    original_df = cleaned_station_data[station]
    interpolated_df = interpolated_station_data[station]

    print(
        f"{station}: "
        f"rows before = {len(original_df):,}, "
        f"rows after = {len(interpolated_df):,}, "
        f"timestamps unchanged = "
        f"{original_df['Datetime'].equals(interpolated_df['Datetime'])}"
    )

# Section 2: Checking variables that were excluded from interpolation
print("\n=== 2. Excluded variables check ===")

excluded_check = []

for station in retained_stations:

    original_df = cleaned_station_data[station]
    interpolated_df = interpolated_station_data[station]

    for column in original_df.columns:

        if column == "Datetime":
            continue

        column_upper = column.upper()

        if "RAIN" in column_upper or "WDR" in column_upper:

            changed = not original_df[column].equals(
                interpolated_df[column]
            )

            excluded_check.append({
                "Station": station,
                "Variable": column,
                "Changed": changed
            })

excluded_check_df = pd.DataFrame(excluded_check)

display(excluded_check_df)

print(
    "Excluded-variable values changed:",
    excluded_check_df["Changed"].any()
)

# Section 3: Checking for missing values (those gaps >48hours)
print("\n=== 3. Remaining missing values ===")

remaining_missing = []

for station in retained_stations:

    df = interpolated_station_data[station]

    for column in df.columns:

        if column == "Datetime":
            continue

        missing_count = df[column].isna().sum()

        remaining_missing.append({
            "Station": station,
            "Variable": column,
            "Remaining_Missing_Values": missing_count
        })

remaining_missing_df = pd.DataFrame(remaining_missing)

display(
    remaining_missing_df[
        remaining_missing_df["Remaining_Missing_Values"] > 0
    ].sort_values(
        "Remaining_Missing_Values",
        ascending=False
    )
)

# Section 4: Missing PM2.5 values for long gaps 
print("\n=== 4. PM2.5 missing values after interpolation ===")

pm25_missing_check = []

for station in retained_stations:

    df = interpolated_station_data[station]

    pm25_columns = [
        column
        for column in df.columns
        if "PM2.5" in column.upper()
    ]

    for column in pm25_columns:

        pm25_missing_check.append({
            "Station": station,
            "Variable": column,
            "Remaining_Missing_PM2.5": df[column].isna().sum()
        })

pm25_missing_check_df = pd.DataFrame(pm25_missing_check)

display(pm25_missing_check_df)


print("\nPost-interpolation verification completed.")

=== 1. Timestamp and row-count check ===
Camden: rows before = 57,384, rows after = 57,384, timestamps unchanged = True
Liverpool: rows before = 57,384, rows after = 57,384, timestamps unchanged = True
Parramatta North: rows before = 57,384, rows after = 57,384, timestamps unchanged = True
Penrith: rows before = 57,384, rows after = 57,384, timestamps unchanged = True
Prospect: rows before = 57,384, rows after = 57,384, timestamps unchanged = True
Randwick: rows before = 57,384, rows after = 57,384, timestamps unchanged = True
Rozelle: rows before = 57,384, rows after = 57,384, timestamps unchanged = True

=== 2. Excluded variables check ===


,Station,Variable,Changed
0,Camden,CAMDEN RAIN 1h average [mm/m²],False
1,Camden,CAMDEN WDR 1h average [°],False
2,Liverpool,LIVERPOOL RAIN 1h average [mm/m²],False
3,Liverpool,LIVERPOOL WDR 1h average [°],False
4,Parramatta North,PARRAMATTA NORTH RAIN 1h average [mm/m²],False
5,Parramatta North,PARRAMATTA NORTH WDR 1h average [°],False
6,Penrith,PENRITH RAIN 1h average [mm/m²],False
7,Penrith,PENRITH WDR 1h average [°],False
8,Prospect,PROSPECT WDR 1h average [°],False
9,Randwick,RANDWICK RAIN 1h average [mm/m²],False


Excluded-variable values changed: False

=== 3. Remaining missing values ===


,Station,Variable,Remaining_Missing_Values
66,Randwick,RANDWICK RAIN 1h average [mm/m²],24524
30,Parramatta North,PARRAMATTA NORTH RAIN 1h average [mm/m²],20625
8,Camden,CAMDEN RAIN 1h average [mm/m²],15164
18,Liverpool,LIVERPOOL RAIN 1h average [mm/m²],15035
14,Liverpool,LIVERPOOL CO 1h average [ppm],15005
38,Penrith,PENRITH CO 1h average [ppm],13478
36,Penrith,PENRITH HUMID 1h average [%],5074
35,Penrith,PENRITH TEMP 1h average [°C],5055
39,Penrith,PENRITH NO 1h average [pphm],5002
37,Penrith,PENRITH NO2 1h average [pphm],5001



=== 4. PM2.5 missing values after interpolation ===


,Station,Variable,Remaining_Missing_PM2.5
0,Camden,CAMDEN PM2.5 1h average [µg/m³],1252
1,Liverpool,LIVERPOOL PM2.5 1h average [µg/m³],1063
2,Parramatta North,PARRAMATTA NORTH PM2.5 1h average [µg/m³],1467
3,Penrith,PENRITH PM2.5 1h average [µg/m³],4713
4,Prospect,PROSPECT PM2.5 1h average [µg/m³],2853
5,Randwick,RANDWICK PM2.5 1h average [µg/m³],557
6,Rozelle,ROZELLE PM2.5 1h average [µg/m³],420



Post-interpolation verification completed.


##### 2.5.8 Final Missing-Value Summary

After short-gap interpolation, the remaining missing values are summarised for each station and variable. These remaining gaps primarily represent prolonged periods of unavailable observations that were intentionally not interpolated. The summary is retained to document the completeness of the datasets used for subsequent analysis.

In [25]:
# Summarise missing values remaining after short-gap interpolation.

final_missing_summary = []

for station in retained_stations:

    df = interpolated_station_data[station]

    for column in df.columns:

        if column == "Datetime":
            continue

        missing_count = df[column].isna().sum()
        total_count = len(df)
        missing_percentage = (missing_count / total_count) * 100

        final_missing_summary.append({
            "Station": station,
            "Variable": column,
            "Remaining_Missing_Values": missing_count,
            "Total_Values": total_count,
            "Missing_Percentage": missing_percentage
        })

final_missing_summary_df = pd.DataFrame(final_missing_summary)

display(
    final_missing_summary_df.sort_values(
        "Missing_Percentage",
        ascending=False
    )
)

print("Final missing-value summary completed.")

,Station,Variable,Remaining_Missing_Values,Total_Values,Missing_Percentage
66,Randwick,RANDWICK RAIN 1h average [mm/m²],24524,57384,42.736651
30,Parramatta North,PARRAMATTA NORTH RAIN 1h average [mm/m²],20625,57384,35.942074
8,Camden,CAMDEN RAIN 1h average [mm/m²],15164,57384,26.425484
18,Liverpool,LIVERPOOL RAIN 1h average [mm/m²],15035,57384,26.200683
14,Liverpool,LIVERPOOL CO 1h average [ppm],15005,57384,26.148404
38,Penrith,PENRITH CO 1h average [ppm],13478,57384,23.487383
36,Penrith,PENRITH HUMID 1h average [%],5074,57384,8.842186
35,Penrith,PENRITH TEMP 1h average [°C],5055,57384,8.809076
39,Penrith,PENRITH NO 1h average [pphm],5002,57384,8.716715
37,Penrith,PENRITH NO2 1h average [pphm],5001,57384,8.714973


Final missing-value summary completed.


In [26]:
# Save the final missing-value summary for documentation and later reference.

final_missing_summary_path = tables_dir / "final_missing_value_summary.csv"

final_missing_summary_df.to_csv(
    final_missing_summary_path,
    index=False
)

print(f"Saved final missing-value summary to: {final_missing_summary_path}")

Saved final missing-value summary to: ..\results\tables\final_missing_value_summary.csv


#### 2.6 Final Dataset Preparation and Checks

##### 2.6.1 Selection of Analysis Variables

The final analysis datasets are restricted to the variables defined for the study. PM2.5 is retained as the target variable, while temperature, humidity, wind speed, wind direction, rainfall, PM10, NO2, CO, NO, OZONE, and SO2 are retained as potential predictor variables. The timestamp is also retained to preserve the temporal structure of the observations.

In [35]:
# Select the final analysis variables using exact variable-name matching.
# This prevents variables such as "NO" from matching text inside
# station names such as "NORTH".

import re

analysis_variables = [
    "PM2.5",
    "TEMP",
    "HUMID",
    "WSP",
    "WDR",
    "RAIN",
    "PM10",
    "NO2",
    "CO",
    "NO",
    "OZONE",
    "SO2"
]

final_station_data = {}

for station in retained_stations:

    df = interpolated_station_data[station].copy()

    selected_columns = ["Datetime"]

    for variable in analysis_variables:

        matching_columns = []

        for column in df.columns:

            if column == "Datetime":
                continue

            column_upper = column.upper()

            # Match the variable as a separate token.
            # This prevents "NO" from matching "NORTH".
            pattern = rf"(?<![A-Z0-9]){re.escape(variable.upper())}(?![A-Z0-9])"

            if re.search(pattern, column_upper):
                matching_columns.append(column)

        if len(matching_columns) == 1:
            selected_columns.append(matching_columns[0])

        elif len(matching_columns) > 1:
            print(
                f"WARNING: Multiple matches for {station} - {variable}: "
                f"{matching_columns}"
            )

        else:
            print(
                f"\nNOTE: {station} - {variable} is not available.\n"
            )

    final_station_data[station] = df[selected_columns].copy()

    print(
        f"{station}: "
        f"{final_station_data[station].shape[0]:,} rows, "
        f"{final_station_data[station].shape[1] - 1} analysis variables"
    )

print("\nFinal analysis variables selected.")


NOTE: Camden - SO2 is not available.

Camden: 57,384 rows, 11 analysis variables
Liverpool: 57,384 rows, 12 analysis variables
Parramatta North: 57,384 rows, 12 analysis variables
Penrith: 57,384 rows, 12 analysis variables

NOTE: Prospect - RAIN is not available.

Prospect: 57,384 rows, 11 analysis variables

NOTE: Randwick - CO is not available.

Randwick: 57,384 rows, 11 analysis variables
Rozelle: 57,384 rows, 12 analysis variables

Final analysis variables selected.


In [33]:
print("Parramatta North selected columns:")

for column in final_station_data["Parramatta North"].columns:
    print(column)

Parramatta North selected columns:
Datetime
PARRAMATTA NORTH PM2.5 1h average [µg/m³]
PARRAMATTA NORTH TEMP 1h average [°C]
PARRAMATTA NORTH HUMID 1h average [%]
PARRAMATTA NORTH WSP 1h average [m/s]
PARRAMATTA NORTH WDR 1h average [°]
PARRAMATTA NORTH RAIN 1h average [mm/m²]
PARRAMATTA NORTH PM10 1h average [µg/m³]
PARRAMATTA NORTH NO2 1h average [pphm]
PARRAMATTA NORTH CO 1h average [ppm]
PARRAMATTA NORTH NO 1h average [pphm]
PARRAMATTA NORTH OZONE 1h average [pphm]
PARRAMATTA NORTH SO2 1h average [pphm]


##### 2.6.2 Duplicate Timestamp Check

The final station datasets are checked for duplicate timestamps to confirm that each station contains at most one observation for each hourly timestamp. This ensures that the final datasets maintain the expected hourly temporal structure.

In [36]:
# Check for duplicate timestamps in the final station datasets.

duplicate_timestamp_summary = []

for station in retained_stations:

    df = final_station_data[station]

    duplicate_count = df["Datetime"].duplicated().sum()

    duplicate_timestamp_summary.append({
        "Station": station,
        "Total_Rows": len(df),
        "Duplicate_Timestamps": duplicate_count
    })

duplicate_timestamp_summary_df = pd.DataFrame(
    duplicate_timestamp_summary
)

display(duplicate_timestamp_summary_df)

print(
    "\nTotal duplicate timestamps across all stations:",
    duplicate_timestamp_summary_df["Duplicate_Timestamps"].sum()
)

,Station,Total_Rows,Duplicate_Timestamps
0,Camden,57384,0
1,Liverpool,57384,0
2,Parramatta North,57384,0
3,Penrith,57384,0
4,Prospect,57384,0
5,Randwick,57384,0
6,Rozelle,57384,0



Total duplicate timestamps across all stations: 0


##### 2.6.3 Final Physical-Validity Check

A final physical-validity check is performed after interpolation to confirm that the cleaned datasets do not contain observations outside the predefined physical ranges. This verifies that the preprocessing steps have not introduced new physically invalid measurements.

In [37]:
# Check for physically invalid values in the final analysis datasets.

final_invalid_summary = []

for station in retained_stations:

    df = final_station_data[station]

    for column in df.columns:

        if column == "Datetime":
            continue

        column_upper = column.upper()

        if "HUMID" in column_upper:
            invalid = (df[column] < 0) | (df[column] > 100)

        elif "PM2.5" in column_upper or "PM10" in column_upper:
            invalid = df[column] < 0

        elif "WDR" in column_upper:
            invalid = (df[column] < 0) | (df[column] > 360)

        elif "WSP" in column_upper or "RAIN" in column_upper:
            invalid = df[column] < 0

        elif any(
            pollutant in column_upper
            for pollutant in ["NO2", "NO", "CO", "OZONE", "SO2"]
        ):
            invalid = df[column] < 0

        else:
            continue

        invalid_count = invalid.sum()

        if invalid_count > 0:
            final_invalid_summary.append({
                "Station": station,
                "Variable": column,
                "Invalid_Values": invalid_count
            })

final_invalid_summary_df = pd.DataFrame(final_invalid_summary)

if final_invalid_summary_df.empty:

    print("No physically invalid values detected.")

else:

    display(
        final_invalid_summary_df.sort_values(
            "Invalid_Values",
            ascending=False
        )
    )

No physically invalid values detected.


##### 2.6.4 Final Data-Range Check

The minimum and maximum observed values of the final analysis variables are examined for each station. This provides a final descriptive check of the data ranges before exploratory analysis. The check is used to identify potentially unusual distributions for further investigation during EDA rather than automatically removing observations.

In [38]:
# Summarise the observed ranges of the final analysis variables.

final_range_summary = []

for station in retained_stations:

    df = final_station_data[station]

    for column in df.columns:

        if column == "Datetime":
            continue

        final_range_summary.append({
            "Station": station,
            "Variable": column,
            "Minimum": df[column].min(),
            "Maximum": df[column].max()
        })

final_range_summary_df = pd.DataFrame(final_range_summary)

display(
    final_range_summary_df.sort_values(
        ["Station", "Variable"]
    )
)

print("Final data-range check completed.")

,Station,Variable,Minimum,Maximum
8,Camden,CAMDEN CO 1h average [ppm],0.0,4.9
2,Camden,CAMDEN HUMID 1h average [%],9.3,100.0
9,Camden,CAMDEN NO 1h average [pphm],0.0,6.0
7,Camden,CAMDEN NO2 1h average [pphm],0.0,4.1
10,Camden,CAMDEN OZONE 1h average [pphm],0.0,11.5
6,Camden,CAMDEN PM10 1h average [µg/m³],0.0,1199.0
0,Camden,CAMDEN PM2.5 1h average [µg/m³],0.0,513.0
5,Camden,CAMDEN RAIN 1h average [mm/m²],0.0,60.6
1,Camden,CAMDEN TEMP 1h average [°C],-1.6,45.2
4,Camden,CAMDEN WDR 1h average [°],0.0,360.0


Final data-range check completed.


###### There are some very large maxima, especially:

###### - Camden PM10: 1,199.0
###### - Liverpool PM10: 1,545.8
###### - Penrith PM10: 14,073.1
###### - Camden PM2.5: 513.0
###### - Randwick PM2.5: 556.3

###### But we are NOT removing these here. This is exactly why we are doing the EDA next: we'll inspect distributions and boxplots and determine whether these are genuine extreme observations, data-quality issues, or values that need special treatment.

##### 2.6.5 Saving Final Station Datasets

The final preprocessed datasets for the retained stations are saved as CSV files for use in subsequent exploratory analysis and modelling. The datasets retain the hourly timestamp structure, selected analysis variables, cleaned physically invalid observations, and interpolated values for eligible short missing gaps.

In [41]:
# Save the final preprocessed dataset for each retained station.

final_processed_dir = processed_dir.parent / "final"

final_processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

for station in retained_stations:

    df = final_station_data[station].copy()

    output_path = final_processed_dir / f"{station}_final_preprocessed.csv"

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"{station}: saved to {output_path}"
    )

print("\nAll final station datasets have been saved.")

Camden: saved to ..\data\final\Camden_final_preprocessed.csv
Liverpool: saved to ..\data\final\Liverpool_final_preprocessed.csv
Parramatta North: saved to ..\data\final\Parramatta North_final_preprocessed.csv
Penrith: saved to ..\data\final\Penrith_final_preprocessed.csv
Prospect: saved to ..\data\final\Prospect_final_preprocessed.csv
Randwick: saved to ..\data\final\Randwick_final_preprocessed.csv
Rozelle: saved to ..\data\final\Rozelle_final_preprocessed.csv

All final station datasets have been saved.


#### 2.7 Handling of Remaining Missing Values

After short-gap interpolation, some missing values remain because they belong to prolonged gaps or variables that were excluded from ordinary linear interpolation (Wind direction, Rain). These remaining missing values are not filled artificially at this preprocessing stage.

The retained missing values will be considered during Exploratory Data Analysis (EDA) to assess their temporal and station-level patterns. For subsequent machine learning, rows with missing target or predictor values will be handled when constructing the modelling datasets, according to the requirements of each modelling approach.

This approach avoids introducing potentially unreliable estimates across prolonged periods of unavailable observations while preserving the original missing-data information for further analysis.

In [43]:
# Summarise the remaining missing values in the final station datasets.

remaining_missing_summary = []

for station in retained_stations:

    df = final_station_data[station]

    for column in df.columns:

        if column == "Datetime":
            continue

        missing_count = df[column].isna().sum()

        if missing_count > 0:
            remaining_missing_summary.append({
                "Station": station,
                "Variable": column,
                "Missing_Values": missing_count,
                "Total_Values": len(df),
                "Missing_Percentage": (
                    missing_count / len(df)
                ) * 100
            })

remaining_missing_summary_df = pd.DataFrame(
    remaining_missing_summary
)

display(
    remaining_missing_summary_df.sort_values(
        "Missing_Percentage",
        ascending=False
    )
)

,Station,Variable,Missing_Values,Total_Values,Missing_Percentage
59,Randwick,RANDWICK RAIN 1h average [mm/m²],24524,57384,42.736651
25,Parramatta North,PARRAMATTA NORTH RAIN 1h average [mm/m²],20625,57384,35.942074
5,Camden,CAMDEN RAIN 1h average [mm/m²],15164,57384,26.425484
16,Liverpool,LIVERPOOL RAIN 1h average [mm/m²],15035,57384,26.200683
19,Liverpool,LIVERPOOL CO 1h average [ppm],15005,57384,26.148404
39,Penrith,PENRITH CO 1h average [ppm],13478,57384,23.487383
33,Penrith,PENRITH HUMID 1h average [%],5074,57384,8.842186
32,Penrith,PENRITH TEMP 1h average [°C],5055,57384,8.809076
40,Penrith,PENRITH NO 1h average [pphm],5002,57384,8.716715
38,Penrith,PENRITH NO2 1h average [pphm],5001,57384,8.714973


#### 2.8 Preprocessing Summary

The preprocessing stage combined the available air-quality and meteorological variables for the seven retained monitoring stations. Physically invalid and suspicious negative measurements were converted to missing values rather than deleting the corresponding timestamps. Missing-value patterns were then assessed, and linear interpolation was applied only to short gaps of up to 48 consecutive hours for suitable continuous variables. Rainfall and wind direction were excluded from ordinary linear interpolation, while prolonged missing-value gaps were retained as missing.

The final datasets were checked for duplicate timestamps and physically invalid values. The resulting station-level datasets preserve the hourly temporal structure and contain the selected analysis variables available at each station. These final preprocessed datasets are saved for use in the exploratory data analysis stage.

In [45]:
print("Notebook 02 preprocessing completed successfully.")
print(f"Final datasets saved for {len(retained_stations)} retained stations.")
print("Next stage: Exploratory Data Analysis (Notebook 03).")
print("Handling of the long-gap missing values/WindDir and Rain will be done in EDA")

Notebook 02 preprocessing completed successfully.
Final datasets saved for 7 retained stations.
Next stage: Exploratory Data Analysis (Notebook 03).
Handling of the long-gap missing values/WindDir and Rain will be done in EDA
